# ReLiC Quick Start

A simple guide to using ReLiC for atmospheric retrieval of transiting exoplanets

## 1. Prepare your data

Use data reduction pipelines to process the raw data until you have the spectral light curves. 

ReLiC uses spectral light curves as input data. 

The spectral light curves of each observation should be stored in an HDF5 file containing:
- `bjd_tdb` — 1D time array
- `wavelength` — 1D wavelength array (bin centers) in μm
- `flux` — 2D flux array (n_wavelength × n_time)
- `flux_err` — 2D error array (n_wavelength × n_time)
- `wavelength_bins` (optional) — 2D wavelength array (n_wavelength × 2). When not provided, they will be calculated using numpy.diff(wavelength)


## 2. Prepare your configuration file

Configuration is done via a TOML file, defining everything that ReLiC requires.

A minimal config includes `[PATH]`, `[STAR]`, `[PLANET]`, `[ATMOSPHERE]`, `[EXOIRIS]`, `[PRIORS]`, and `[SAMPLER]` sections. See `config/` for examples. 

Below are explanations to a minimal configuration file. 

```toml
[PATH]
lightcurve_files = [ "data1.h5", "data2.h5", ] # input spectral light curves
spec_resolving_power_files = [ "NIRCam.dat", "", ] # ascii files of instrumental resolving powers with two columns (wv in micron, resolving_power), necessary for pixel-resolution retrievals. Use empty strings to skip the convolution step if you want.
output_dir = "output/" # where you want to store the outputs

[STAR] 
name        = "HD209458" # name of the host star
teff        = [6091, 50] # stellar effective temperature and error
logg        = [4.45, 0.02] # star logg and error
metal       = [0.01, 0.05] # stellar metallicity and error
radius_rsun = [1.19, 0.02] # stellar radius and error

[PLANET] 
name                    = "HD209458b" # name of the planet
transit_epoch_bjd       = [2455216.405640, 0.000094] # transit epoch in BJD_TDB 
transit_period_d        = [3.52474859, 0.00000038] # transit period in days
transit_duration_d      = [0.127, 0.001] # transit duration in days
radius_rjup             = [1.39, 0.02] # planet radius in Jupiter radius
mass_mjup               = [0.73, 0.04] # planet mass in Jupiter mass
equilibrium_temperature = [1459, 12] # planet equilibrium temperature in kelvin
circular_orbit          = true # Eccentricity will be fixed to zero if true, otherwise two extra free parameters (sqrt(e)cos(w) and sqrt(e)sin(w)) will be added for eccentric orbits

[ATMOSPHERE] # Depending on your atmospheric model, you may have different key-value pairs
model_class              = "IsothermalAtmosphere" # IMPORTANT, the class name of your atmospheric model
wavelength_bounds_micron = [1, 2]
chemical_species         = [ "H2O", "CO", "CO2", "CH4", ]
...

[EXOIRIS]
instruments = [ "JWST/NIRCam_F322W2", "JWST/NIRCam_F444W"] # instrument name for each dataset 
wl_range_micron   = [ [], [] ] # crop useful wavelength range (optional)
time_range_bjd    = [ [], [] ] # crop useful time range (optional)
noise_groups      = [0, 1] # group datasets of same instrument setups so that they share the same jitter factors
n_baselines       = [2, 2] # Define the max degree of polynomial baselines (should be at least 1)
rebin_resolutions = [100, 200] # set to 0 for pixel resolution
noise_model        = "white_marginalized" # four available options ["white_marginalized", "white_profiled", "free_gp", "fixed_gp"]. Please refer to ExoIris document for more information

[PRIORS.TRANSIT] # prior assumption of transit parameters
rho         = ["NP", 1.04, 0.07, 0.7, 1.4, "stellar density", "cgs"]
p           = ["NP", 3.52474859, 0.00000038, 0, inf, "transit period", "days"]
b           = ["NP", 0.5056, 0.0133, 0, 1, "impact parameter", "R_s"]
tc_00       = ["UP", 2459890.15, 2459890.25, -inf, inf, "transit epoch", "BJD_TDB"]

[PRIORS.STAR] # prior assumption of stellar parameters
teff        = ["NP", 6091, 50, 5000, 7000, "stellar effective temperature", "K"]
logg        = ["NP", 4.45, 0.02, 4, 5, "stellar gravity", "log10 cgs"]
metal       = ["NP", 0.01, 0.05, -1, 1, "stellar metallicity", ""]

[PRIORS.NOISE] # prior assumption of noise parameters
sigma_m_00  = ["UP", 0.2, 5, -inf, inf, "noise factor 00", ""]
sigma_m_01  = ["UP", 0.2, 5, -inf, inf, "noise factor 01", ""]

[PRIORS.ATMOSPHERE] # prior assumption of atmospheric parameters
mp          = ["NP", 0.73, 0.04, 0.53, 0.93, "planet mass", "M_j"]
temp        = ["UP", 100, 3000, -inf, inf, "temperature", "K"]
m2h         = ["UP", -3, 3, -inf, inf, "atmospheric metallicity", "log10"]
c2o         = ["UP", 0.1, 2, -inf, inf, "C/O ratio", " "]
...

[SAMPLER]
method      = "dynesty" # two options: ["dynesty", "emcee"]
npools      = 60 # number of processes for mulprocessing parallelization
niter_white = 1000 # number of iterations for white-light curve fit
nwalkers    = 60 # number of MCMC walkers (white-light curve fit always use MCMC)
bound       = "multi" # bound definition of dynesty
sample      = "rwalk" # sample method of dynesty
n_live_points = 240  # number of live points of dynesty
dlogz_init  = 0.1 # dlogz stop criterion of dynesty
n_effective = 400 # minimum required effective samples. This is the second stop criterion once dlogz_init is satisfied.

save_checkpoint = true  # set to true if you want to resume in the future
resume          = false # set to true if you want to resume from the latest checkpoint

```

## Usage

Multiprocessing is highly recommended even for demo purposes. Set environment variables for thread control:

```python
import os
os.environ["OMP_NUM_THREADS"] =        "1"
os.environ["OPENBLAS_NUM_THREADS"] =   "1"
os.environ["MKL_NUM_THREADS"] =        "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] =    "1"
os.environ["NUMBA_NUM_THREADS"] =      "1" 
os.environ['NUMBA_THREADING_LAYER'] = 'workqueue'   
from multiprocessing import Pool
```

### Nested Sampling Pipeline

```bash
python example_scripts/pipeline_ns.py -c config/my_target.toml
```

This script:
1. Initializes `ReLic` and `PlotFigure`
2. Fits white light curves to validate systematics
3. Tests likelihood evaluation
4. Runs `dynesty` nested sampling (with dynamic live points)
5. Generates diagnostic plots:
   - `white_fit.png` — white light curve fits
   - `fluxes.png` — 2D spectral flux maps
   - `ldprofiles.png` — limb-darkening profiles
   - `corners.pdf` — posterior corner plot
   - `residuals.png` — best-fit residuals
   - Transmission/emission spectra

### Batch Execution

```bash
bash example_scripts/run.sh
```

To run in background with logging:

```bash
nohup bash example_scripts/run.sh > output.log 2>&1 &
tail -f output.log
```

### Using Jupyter Notebooks

The scripts auto-detect if they are running in IPython and use a default config. You can step through the pipeline interactively:

```python
from relic.core import ReLic
from relic.plots import PlotFigure

relic = ReLic("config/HD209458b-jwst-pix-tp6fastchem.toml")
pfig  = PlotFigure(relic)
```

---

## Atmosphere Models

ReLiC provides several built-in atmospheric models. All inherit from `BaseAtmosphere` and must implement `__call__(pv) -> ndarray` returning transit depths.

| Model Class | Chemistry | T–P Profile | Requires |
|---|---|---|---|
| `IsothermalEqChem` | Equilibrium (petitRADTRANS tables) | Isothermal | — |
| `IsothermalFreeChem` | Free chemistry (log10 mass fractions) | Isothermal | — |
| `TP6EqChem` | Equilibrium (petitRADTRANS tables) | 6-parameter Madhusudhan profile | — |
| `TP6FreeChem` | Free chemistry (log10 mass fractions) | 6-parameter Madhusudhan profile | — |
| `TP6FastChem` | FastCHEM chemical equilibrium | 6-parameter Madhusudhan profile | `pyfastchem` + input files |
| `TP6FastChem_SO2` | FastCHEM + free SO2 abundance | 6-parameter Madhusudhan profile | `pyfastchem` + input files |
| `GuillotEQChem` | Equilibrium (petitRADTRANS tables) | Guillot (2010) analytic | — |

To implement a custom model, subclass `BaseAtmosphere`:

```python
from relic.atmosphere import BaseAtmosphere

class MyAtmos(BaseAtmosphere):
    def __init__(self, config):
        super().__init__(config)
        # Set self.pressures_bar, self.wavelengths
        # Set self._sl_atm via config or user

    def __call__(self, pv):
        # Compute transmission/emission spectrum
        # Return transit depth array (n_wl,)
        pass
```

Then specify `model_class = "MyAtmos"` in the config file.

---

## Noise Models

Four noise models are available, set via `noise_model` in `[EXOIRIS]`:

| Model | Description |
|---|---|
| `white_profiled` | Per-point white noise with profiled (marginalized) baseline polynomial coefficients. Fast. |
| `white_marginalized` | Per-point white noise with analytically marginalized baselines. Recommended. |
| `fixed_gp` | Gaussian Process with fixed hyperparameters (optimized once). |
| `free_gp` | Gaussian Process with free hyperparameters (sampled jointly). Uses `celerite2` Matern-3/2 kernel. |